# Tools



### Tools are a pairing of : 

* A schema : name of tool, description of function, and/or argument definitions (often as a JSON schema)
* A function or coroutine to execute

In [19]:
import os

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why is sky blue? explain in one sentence.")
response

AIMessage(content="<think>\nOkay, the user is asking why the sky is blue. Let me start by recalling what I know about this. I think it has to do with sunlight and the atmosphere. Oh right, Rayleigh scattering. So sunlight is white but made up of different colors, each with different wavelengths. Shorter wavelengths scatter more when they hit molecules in the atmosphere.\n\nWait, blue light has a shorter wavelength compared to red or yellow, right? So when sunlight enters the atmosphere, the blue light gets scattered in all directions by the gases and particles. This scattered blue light is what we see when we look up. But why not violet? I remember that violet has an even shorter wavelength than blue. Maybe the sun emits more blue light than violet, or our eyes are more sensitive to blue. Also, some of the violet is absorbed by the upper atmosphere. So the dominant color we perceive is blue. \n\nPutting it all together, the sky appears blue because the Earth's atmosphere scatters short

In [16]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """This function returns the weather of a given location."""
    return "The weather in {location} is cloudy now but will be sunny later"

model_with_tools = model.bind_tools([get_weather])
response = model_with_tools.invoke("What is the weather in Pune?")
print(response.tool_calls)

for tool in response.tool_calls:
    print(tool['name'])
    print(tool['args'])

[{'name': 'get_weather', 'args': {'location': 'Pune'}, 'id': 'vv581wrp6', 'type': 'tool_call'}]
get_weather
{'location': 'Pune'}


### Tool Execution loops

In [18]:
messages = [{"role": "user", "content": "What is the weather in Cedar Rapids?"}]
response = model_with_tools.invoke(messages)
messages.append(response)

for tool_call in response.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
print(final_response.text)
messages


The weather in Cedar Rapids is currently cloudy but will become sunny later. Enjoy the improving conditions! ☀️


[{'role': 'user', 'content': 'What is the weather in Cedar Rapids?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Cedar Rapids. I need to use the get_weather function. The function requires a location parameter. Cedar Rapids is the location they mentioned. So I\'ll call the function with location set to "Cedar Rapids". Let me make sure there are no typos. Everything looks good. Time to format the tool call correctly.\n', 'tool_calls': [{'id': 'w1bvfnn6j', 'function': {'arguments': '{"location":"Cedar Rapids"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 157, 'total_tokens': 257, 'completion_time': 0.148110062, 'completion_tokens_details': {'reasoning_tokens': 74}, 'prompt_time': 0.006595326, 'prompt_tokens_details': None, 'queue_time': 0.069642005, 'total_time': 0.154705388}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76c